# Budget Auction — Binary Search Notebook

This notebook covers both Experiment 1 (public-state) and Experiment 2 (multi-good). Pick the experiment section that matches your setup, set `VARIANT`, and run top-to-bottom. Results are fault-tolerantly saved after every trial.

**Experiment 1** — Public-state budget auction (2 buyers, 1 good, T=5)

**Experiment 2** — Multi-good privately correlated budget auction (2 buyers, 2 goods, T=5)

---
# Experiment 1 Section

# Budget Auction — Binary Search for Optimal Profit Goal (Exp 1 (Public-State Budget))

This notebook implements a binary search over `profit_goal` to find the highest revenue target that yields a mechanism satisfying **IC regret ≤ 0.05** and **IR loss ≤ 0.05** on held-out evaluation keys.

**Key idea:** at each bisection step, we train with full epochs and record ALL mechanism checkpoints during training (not just the final one). We then evaluate every checkpoint on held-out keys. If *any* checkpoint passes both thresholds, the profit_goal is declared feasible.

This avoids discarding good intermediate solutions that the optimizer may subsequently "over-train away".

**Run one variant (A, B, or C) per session.** Set `VARIANT` below, then run all cells top-to-bottom.

In [ ]:
# ── Environment Selection ────────────────────────────────────────────────────
# 'colab'  → installs packages (JAX CUDA 12) and mounts Google Drive
# 'local'  → skips install; optionally changes working directory
# ─────────────────────────────────────────────────────────────────────────────
RUN_ENV = 'colab'   # ← change to 'local' for a local GPU run

# For local runs: absolute path to the Budget Auction project root.
# Set to None if you launch Jupyter from that directory.
LOCAL_PROJECT_DIR = None
# Windows example:
#   LOCAL_PROJECT_DIR = r'E:\Research\code\Dynamic Mechanism Design\Budget Auction'
# Linux / Mac example:
#   LOCAL_PROJECT_DIR = '/home/user/research/budget_auction'


In [ ]:
import os, sys, subprocess

if RUN_ENV == 'colab':
    print('Installing JAX CUDA 12 + dependencies ...')
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'uninstall', '-y',
         'jax', 'jaxlib', 'jax-cuda12-plugin', 'jax-cuda12-pjrt'],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '-qU',
         'jax[cuda12]', 'dm-haiku', 'optax', 'flax'])
    print('Install finished.')
    print('IMPORTANT: Runtime → Restart session, then re-run from this cell.')
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        os.chdir('/content/drive/MyDrive/My Research/Dynamic Mechanism Design/Budget Auction')
        print(f'Working directory: {os.getcwd()}')
    except ImportError:
        print('(Drive mount skipped — restart session first if on Colab.)')
elif RUN_ENV == 'local':
    if LOCAL_PROJECT_DIR is not None:
        os.chdir(LOCAL_PROJECT_DIR)
    print(f'Working directory: {os.getcwd()}')
    print('Local run — package install skipped.')
    print('One-time install (Windows/CPU): pip install "jax[cpu]" dm-haiku optax flax')
    print('One-time install (Linux/WSL2 CUDA 12): pip install "jax[cuda12]" dm-haiku optax flax')
    print('GPU check: python -c "import jax; print(jax.devices())"')
    print('NOTE: Use the Python 3.13 (JAX) kernel — JAX does not support Python 3.14.')
else:
    raise ValueError(f'Unknown RUN_ENV: {RUN_ENV!r}  — set to "colab" or "local"')


In [ ]:
import jax
import jax.numpy as jnp
import haiku as hk
import optax
import matplotlib.pyplot as plt
import pickle, json, os, time

from budget_auction.core import (
    BudgetSimulator, BoundedContPolicy,
    build_bounded_policy_net, plot_loss_dict,
)
from budget_auction.mechanisms.budget_mechanism import BudgetMechanism
from budget_auction.mechanisms.input_adapters import VARIANT_NETWORK
from budget_auction.envs.public_budget_env import create_public_budget_auction
from budget_auction.train.run_acceptance import nested_GD_budget
from budget_auction.train.evaluate import eval_budget_mechanism_multikey, print_eval_report
from budget_auction.train.binary_search import (
    DEFAULT_BS_CFG, train_with_checkpoints,
    evaluate_checkpoint_list, run_binary_search,
)

print('JAX devices:', jax.devices())


## Configuration

Set `VARIANT` to the mechanism input variant you want to search over. Run one variant per session (each takes ~6-10 hours for a full binary search).

The base hyperparameters below are seeded from the HP search results (Stage 1 winner: `lr_theta=3e-4`, `lr_w=1e-3`; Stage 2: moderate weights). Override `hp_json_path` to load your own HP search results.

In [ ]:
VARIANT  = 'A'   # 'A' | 'B' | 'C'  ← change this per run
SEED     = 42
TIME_BUDGET_HOURS = 11.5   # total wall-clock budget

# ---- Set FORCE_RESTART=True to ignore existing results and start fresh ----
# Use a different bs_save_dir below when changing configs, so old results
# are not overwritten.  FORCE_RESTART=True with the same dir discards them.
FORCE_RESTART = False

# ---- Load HP search best config (optional) ----
hp_json_path = os.path.join('outputs/public_budget/hparam_search', 'best_hparams.json')
if os.path.exists(hp_json_path):
    with open(hp_json_path) as _f:
        _hp = json.load(_f)
    _opt = _hp.get('optimal_config', {})
    print(f'Loaded HP search config from: {hp_json_path}')
    print('  Optimal config:', _opt)
else:
    _opt = {}
    print(f'HP search file not found at {hp_json_path} — using defaults.')

# ---- Binary search config ----
# Seeded from HP search; override any key as needed.
bs_cfg = {
    **DEFAULT_BS_CFG,
    # Learning rates (from HP search Stage 1)
    'lr_theta':        _opt.get('lr_theta',        3e-4),
    'lr_w':            _opt.get('lr_w',            1e-3),
    # Loss weights at pg_base (from HP search Stage 2)
    'alpha_base':      _opt.get('alpha_weight',    1e5),
    'beta_weight':     _opt.get('beta_weight',     500.0),
    # Inner oracle steps (from HP search Stage 3)
    'num_inner_steps': _opt.get('num_inner_steps', 300),
    # Full training epochs per binary-search trial
    'num_outer_epochs': 2000,
    # Checkpoint recording
    'checkpoint_thresh':    0.15,   # record when train IC+IR both ≤ this
    'checkpoint_interval':  50,     # check every N outer epochs
    # Binary search range
    'pg_lo':   0.8,
    'pg_hi':   2.0,
    'pg_base': 0.8,   # reference point for alpha scaling
    'pg_tol':  0.1,
    # Feasibility thresholds (eval)
    'ic_thresh':  0.05,
    # Revenue gap threshold: checkpoint must achieve revenue >= profit_goal - gap_thresh.
    # Prevents a low-revenue IC solution from falsely pushing pg_lo upward.
    # Default matches pg_tol so 'feasible' means within binary-search resolution.
    'gap_thresh': 0.1,
    # IC re-optimisation at eval time.
    # The oracle is re-solved from scratch against each checkpoint mechanism
    # to get an honest (upper-bound) regret rather than reusing the
    # underconverged training oracle.
    'ic_reopt_steps':    1000,   # inner steps per restart
    'ic_reopt_restarts': 3,      # independent random restarts (take max)
    'ic_reopt_lr':       3e-3,   # Adam lr for reopt (independent of training lr_w)
    'ic_reopt_samples':  128,    # MC samples during reopt ascent
    # Evaluation (revenue, IR, budget-viol, final IC)
    'eval_num_keys':    32,
    'eval_num_samples': 256,
}

print(f'Binary search config for Variant {VARIANT}  (FORCE_RESTART={FORCE_RESTART}):')
for _k, _v in bs_cfg.items():
    print(f'  {_k:25s}: {_v}')


## Build Auction Environment

In [ ]:
auction = create_public_budget_auction(
    n_buyers=2, n_goods=1, horizon=5, discount=0.9,
    initial_budget=3.0, seed=SEED,
)
print(f'Auction: {auction.num_buyers}B x {auction.num_goods}G x T={auction.num_horizons}')
print(f'Initial budgets: {auction.initial_budgets}')


## Build Networks

In [ ]:
mechanism_network = hk.without_apply_rng(hk.transform(VARIANT_NETWORK[VARIANT]))
policy_network = build_bounded_policy_net(
    d_priv=auction.d_priv, d_pub=auction.d_pub,
    G=auction.num_goods, hidden=128,
    residual_init_std=1.0,
)
print(f'Variant {VARIANT} networks built.')
print(f'  d_priv={auction.d_priv}  d_pub={auction.d_pub}  G={auction.num_goods}')


In [ ]:
# To start a brand-new search with a different config, change the RUN_TAG below.
# Each unique tag gets its own subdirectory, so old results are never overwritten.
# Leave RUN_TAG='' to use the default directory.
RUN_TAG = ''   # e.g. 'run2', 'higherLR', 'moreSteps'
_bs_base = f'outputs/public_budget/binary_search/variant_{VARIANT}'
bs_save_dir = f'{_bs_base}/{RUN_TAG}' if RUN_TAG else _bs_base
os.makedirs(bs_save_dir, exist_ok=True)
print(f'Save directory: {bs_save_dir}')
print(f'Results will be written to: {bs_save_dir}/bs_results.json (fault-tolerant)')
print(f'Best mechanisms: {bs_save_dir}/best_mech_pg<X>.pkl')
print(f'(Set FORCE_RESTART=True above to discard existing results in this directory)')


## Run Binary Search

This cell runs the full binary search. It will:
1. Bisect the `[pg_lo, pg_hi]` interval until `pg_hi - pg_lo < pg_tol`
2. For each midpoint, train a full mechanism and record checkpoints
3. Evaluate all checkpoints on 32 held-out keys × 256 samples
4. Declare feasibility if any checkpoint achieves IC ≤ 0.05 **AND** revenue ≥ profit_goal − gap_thresh
5. Save results after every trial (safe to interrupt and resume)

Expected number of bisection steps: `ceil(log2((pg_hi-pg_lo)/pg_tol))` ≈ 4-5 trials.
Each trial takes ~30-60 min on GPU. Total wall time: up to 12h.

In [ ]:
bs_summary = run_binary_search(
    auction           = auction,
    mechanism_network = mechanism_network,
    policy_network    = policy_network,
    variant           = VARIANT,
    save_dir          = bs_save_dir,
    cfg               = bs_cfg,
    base_seed         = SEED,
    verbose           = True,
    time_budget_hours = TIME_BUDGET_HOURS,
    force_restart     = FORCE_RESTART,
)

print('\n' + '='*60)
print(f'Binary search complete — Exp 1 (Public-State Budget) Variant {VARIANT}')
print(f'  Highest confirmed feasible profit_goal: {bs_summary["pg_lo"]:.3f}')
print(f'  pg_hi (infeasible upper bound): {bs_summary["pg_hi"]:.3f}')
print(f'  Converged: {bs_summary["converged"]}')
if bs_summary['best_mechanism']:
    _bm = bs_summary['best_mechanism']
    print(f'  Best mechanism: rev={_bm["revenue_eval"]:.4f}  '
          f'IC={_bm["regret_eval"]:.4f}  surplus={_bm.get("buyer_surplus_eval", float("nan")):.4f}')
    print(f'  Saved to: {_bm["mech_pkl"]}')


## Analyse Results

Load the saved trial records and visualise the binary search progression.

In [ ]:
_bs_path = os.path.join(bs_save_dir, 'bs_results.json')
with open(_bs_path) as _f:
    _bs_trials = json.load(_f)
_bs_trials_sorted = sorted(_bs_trials, key=lambda t: t['profit_goal'])

print(f'Loaded {len(_bs_trials)} trial records from {_bs_path}')
print(f'\n{"profit_goal":>12} | {"feasible":>8} | {"best_rev":>9} | {"best_IC":>8} | {"surplus":>9} | {"ckpts":>5}')
print('-' * 65)
for _t in _bs_trials_sorted:
    _flag = 'PASS' if _t.get('any_feasible') else 'fail'
    _rev  = _t.get('best_revenue_eval')       or float('nan')
    _ic   = _t.get('best_regret_eval')         or float('nan')
    _srp  = _t.get('best_buyer_surplus_eval')  or float('nan')
    _nc   = _t.get('num_checkpoints', 0)
    print(f'{_t["profit_goal"]:>12.3f} | {_flag:>8} | {_rev:>9.4f} | {_ic:>8.4f} | {_srp:>9.4f} | {_nc:>5}')


In [ ]:
# Plot binary search progression (one point per bisection trial)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

_pgs  = [t['profit_goal'] for t in _bs_trials_sorted]
_revs = [t.get('best_revenue_eval')       or 0.0 for t in _bs_trials_sorted]
_ics  = [t.get('best_regret_eval')         or 1.0 for t in _bs_trials_sorted]
_srps = [t.get('best_buyer_surplus_eval')  or 0.0 for t in _bs_trials_sorted]
_cols = ['tab:green' if t.get('any_feasible') else 'tab:red' for t in _bs_trials_sorted]

axes[0].scatter(_pgs, _revs, c=_cols, s=80, zorder=3)
axes[0].set_xlabel('profit_goal'); axes[0].set_ylabel('Best feasible revenue')
axes[0].set_title('Revenue vs profit_goal')
axes[0].axhline(0, color='k', lw=0.5)

axes[1].scatter(_pgs, _ics, c=_cols, s=80, zorder=3)
axes[1].axhline(0.05, color='red', linestyle='--', label='IC thresh')
axes[1].set_xlabel('profit_goal'); axes[1].set_ylabel('Best IC regret (eval)')
axes[1].set_title('IC Regret vs profit_goal'); axes[1].legend()

axes[2].scatter(_pgs, _srps, c=_cols, s=80, zorder=3)
axes[2].set_xlabel('profit_goal'); axes[2].set_ylabel('Buyer surplus (eval)')
axes[2].set_title('Buyer Surplus vs profit_goal')

from matplotlib.patches import Patch
_legend = [Patch(color='tab:green', label='feasible'), Patch(color='tab:red', label='infeasible')]
axes[0].legend(handles=_legend)
plt.suptitle(f'Exp 1 (Public-State Budget) — Variant {VARIANT} Binary Search', y=1.02)
plt.tight_layout(); plt.show()


## Convergence Plots per Trial

For each binary-search trial, show:
- **Full training curves** (IC regret, IR loss, profit gap) vs epoch — solid lines
- **Checkpoint markers** — triangles at epochs where training IC+IR ≤ threshold
- **Eval values** at checkpoints — circles (filled=feasible, open=infeasible)
- **Dashed red lines** at the 0.05 feasibility threshold

This lets you see whether the mechanism converges, when good checkpoints appear, and whether training metrics are predictive of eval metrics.

In [ ]:
# Load training curves for all trials
_curves_by_pg = {}
for _t in _bs_trials_sorted:
    _cp = _t.get('training_curves')
    if _cp and os.path.exists(_cp):
        with open(_cp) as _f:
            _curves_by_pg[_t['profit_goal']] = json.load(_f)
    else:
        print(f'  [missing] training_curves for pg={_t["profit_goal"]:.3f}')
print(f'Loaded training curves for {len(_curves_by_pg)} trials.')


In [ ]:
# One figure per trial: 3 panels (IC, IR, profit gap)
# IC panel shows three series:
#   - solid line  : full training regret curve
#   - triangle (▲): training value at each checkpoint
#   - circle (●/x) : re-optimised oracle regret (honest eval); green=pass, red=fail
_THRESH = 0.05
for _pg, _cv in sorted(_curves_by_pg.items()):
    _trial = next(t for t in _bs_trials if abs(t['profit_goal'] - _pg) < 1e-6)
    _feasible_trial = _trial.get('any_feasible', False)
    _title_flag = 'FEASIBLE' if _feasible_trial else 'INFEASIBLE'
    _epochs = list(range(len(_cv['reg'])))

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # ---- Panel data definitions ----
    # (label, full_train_curve, ckpt_train_key, ckpt_eval_key, reopt_key, color)
    # Note: IR loss is always 0 by construction (payment feasibility transform),
    # so we show Buyer Surplus (sum of discounted rewards) instead.
    _panels = [
        ('IC Regret',    _cv['reg'], 'reg_train', 'regret_eval',       'regret_reopt', 'tab:blue'),
        ('Buyer Surplus', None,       'ir_train',  'buyer_surplus_eval', None,          'tab:orange'),
        ('Profit Gap',   _cv['gap'], 'gap_train', None,                 None,          'tab:purple'),
    ]

    for _ax, (_label, _train_curve, _ck_train_key, _ck_eval_key, _ck_reopt_key, _col) in zip(axes, _panels):
        # Full training curve (None for buyer surplus — no training-time metric)
        if _train_curve is not None:
            _ax.plot(_epochs, _train_curve, color=_col, lw=1.2, alpha=0.75, label='train')

        # Feasibility threshold lines: IC panel uses ic_thresh, Gap panel uses gap_thresh
        if _ck_eval_key == 'regret_eval':
            _ax.axhline(_THRESH, color='red', linestyle='--', lw=1.0,
                        alpha=0.8, label=f'IC thresh={_THRESH}')
        if _ck_train_key == 'gap_train':
            _ax.axhline(bs_cfg.get('gap_thresh', 0.1), color='red', linestyle='--', lw=1.0,
                        alpha=0.8, label=f'gap thresh={bs_cfg.get("gap_thresh", 0.1)}')

        # Checkpoint training values (triangles)
        _ck_eps   = [c['epoch']       for c in _cv['checkpoints_train']]
        _ck_train = [c[_ck_train_key] for c in _cv['checkpoints_train']]
        if _ck_eps:
            _ax.scatter(_ck_eps, _ck_train, marker='^', s=60,
                        color=_col, zorder=4, label='ckpt (train)')

        # Checkpoint eval values: re-optimised IC (diamond) or multi-key mean (circle)
        if _ck_eval_key is not None:
            for _er in _cv['checkpoints_eval']:
                _col_pt = 'tab:green' if _er['feasible'] else 'tab:red'
                # Re-opt regret (diamond) — this is what determines feasibility
                if _ck_reopt_key and _er.get(_ck_reopt_key) is not None:
                    _ax.scatter(_er['epoch'], _er[_ck_reopt_key], marker='D', s=70,
                                color=_col_pt, zorder=6,
                                label='reopt IC (pass)' if _er['feasible'] else 'reopt IC (fail)')
                # Multi-key eval mean (circle) — shown for comparison
                if _er.get(_ck_eval_key) is not None:
                    _ax.scatter(_er['epoch'], _er[_ck_eval_key], marker='o', s=45,
                                color=_col_pt, alpha=0.5, zorder=5,
                                label='eval mean')
        elif _ck_eval_key == 'buyer_surplus_eval':
            # Buyer surplus: eval circles only (no training curve, no reopt)
            for _er in _cv['checkpoints_eval']:
                _col_pt = 'tab:green' if _er['feasible'] else 'tab:red'
                if _er.get('buyer_surplus_eval') is not None:
                    _mk = 'o' if _er['feasible'] else 'x'
                    _ax.scatter(_er['epoch'], _er['buyer_surplus_eval'], marker=_mk, s=65,
                                color=_col_pt, zorder=5, label='eval surplus')

        _ax.set_xlabel('Epoch'); _ax.set_ylabel(_label)
        _ax.set_title(_label)
        # Deduplicate legend entries
        _hs, _ls = _ax.get_legend_handles_labels()
        _seen = {}; [_seen.setdefault(_l, _h) for _h, _l in zip(_hs, _ls)]
        _ax.legend(list(_seen.values()), list(_seen.keys()), fontsize=8)

    plt.suptitle(f'Exp 1 (Public-State Budget) Variant {VARIANT} | pg={_pg:.3f} — {_title_flag}', y=1.02)
    plt.tight_layout(); plt.show()


## Load and Evaluate Best Mechanism

In [ ]:
_bm_info = bs_summary.get('best_mechanism') if 'bs_summary' in dir() else None

# Alternatively, find the best pkl from trial records
if _bm_info is None:
    _feas = [t for t in _bs_trials if t.get('any_feasible') and t.get('best_mech_pkl')]
    _feas.sort(key=lambda t: t.get('best_revenue_eval') or 0.0, reverse=True)
    _bm_info = _feas[0] if _feas else None

if _bm_info is None:
    print('No feasible mechanism found yet — search may need more iterations.')
else:
    _pkl_path = _bm_info.get('mech_pkl') or _bm_info.get('best_mech_pkl')
    print(f'Loading best mechanism from: {_pkl_path}')
    with open(_pkl_path, 'rb') as _f:
        _best = pickle.load(_f)

    _best_mnet = hk.without_apply_rng(hk.transform(VARIANT_NETWORK[VARIANT]))
    _best_pnet = build_bounded_policy_net(
        d_priv=auction.d_priv, d_pub=auction.d_pub,
        G=auction.num_goods, hidden=128, residual_init_std=1.0,
    )

    _eval_keys = jax.random.split(jax.random.PRNGKey(55555), 64)
    _final_ev = eval_budget_mechanism_multikey(
        auction=auction, mechanism_network=_best_mnet,
        mechanism_params=_best['mechanism_params'],
        profit_goal=_best['profit_goal'],
        num_samples=512, keys=_eval_keys,
        policy_network=_best_pnet, policy_params=_best['policy_params'],
    )
    print_eval_report(_final_ev, prefix=f'Exp 1 (Public-State Budget) Variant {VARIANT} Best: ')
    print(f'  profit_goal: {_best["profit_goal"]:.3f}')
    print(f'  epoch found: {_best["epoch"]}')


## Save Summary

In [ ]:
_summary_path = os.path.join(f'outputs/public_budget/binary_search/variant_{VARIANT}', f'variant_{VARIANT}_summary.json')
if _bm_info is not None:
    _out = {
        'variant':         VARIANT,
        'exp':             'Exp 1 (Public-State Budget)',
        'pg_lo':           bs_summary.get('pg_lo'),
        'pg_hi':           bs_summary.get('pg_hi'),
        'converged':       bs_summary.get('converged'),
        'total_time_h':    bs_summary.get('total_time_h'),
        'best_mechanism':  {
            'profit_goal':    _best['profit_goal'],
            'epoch':          _best['epoch'],
            'revenue_eval':   float(_final_ev['profit_stats']['mean']),
            'revenue_std':    float(_final_ev['profit_stats']['std']),
            'regret_eval':    float(_final_ev.get('regret_stats', {}).get('mean', float('nan'))),
            'buyer_surplus_eval': float(_final_ev['buyer_surplus_stats']['mean']),
            'budget_viol':    float(_final_ev['budget_viol_stats']['mean']),
            'ir_viol':        float(_final_ev['ir_viol_stats']['mean']),
        },
        'bs_cfg': {k: v for k, v in bs_cfg.items()},
    }
    with open(_summary_path, 'w') as _f:
        json.dump(_out, _f, indent=2)
    print(f'Summary saved to: {_summary_path}')
else:
    print('No feasible mechanism — nothing to save.')


---
# Experiment 2 Section

(Clear GPU memory / restart runtime before switching experiments)

# Budget Auction — Binary Search for Optimal Profit Goal (Exp 2 (Multi-Good Privately Correlated))

This notebook implements a binary search over `profit_goal` to find the highest revenue target that yields a mechanism satisfying **IC regret ≤ 0.05** and **IR loss ≤ 0.05** on held-out evaluation keys.

**Key idea:** at each bisection step, we train with full epochs and record ALL mechanism checkpoints during training (not just the final one). We then evaluate every checkpoint on held-out keys. If *any* checkpoint passes both thresholds, the profit_goal is declared feasible.

This avoids discarding good intermediate solutions that the optimizer may subsequently "over-train away".

**Run one variant (A, B, or C) per session.** Set `VARIANT` below, then run all cells top-to-bottom.

In [ ]:
# ── Environment Selection ────────────────────────────────────────────────────
# 'colab'  → installs packages (JAX CUDA 12) and mounts Google Drive
# 'local'  → skips install; optionally changes working directory
# ─────────────────────────────────────────────────────────────────────────────
RUN_ENV = 'colab'   # ← change to 'local' for a local GPU run

# For local runs: absolute path to the Budget Auction project root.
# Set to None if you launch Jupyter from that directory.
LOCAL_PROJECT_DIR = None
# Windows example:
#   LOCAL_PROJECT_DIR = r'E:\Research\code\Dynamic Mechanism Design\Budget Auction'
# Linux / Mac example:
#   LOCAL_PROJECT_DIR = '/home/user/research/budget_auction'


In [ ]:
import os, sys, subprocess

if RUN_ENV == 'colab':
    print('Installing JAX CUDA 12 + dependencies ...')
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'uninstall', '-y',
         'jax', 'jaxlib', 'jax-cuda12-plugin', 'jax-cuda12-pjrt'],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '-qU',
         'jax[cuda12]', 'dm-haiku', 'optax', 'flax'])
    print('Install finished.')
    print('IMPORTANT: Runtime → Restart session, then re-run from this cell.')
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        os.chdir('/content/drive/MyDrive/My Research/Dynamic Mechanism Design/Budget Auction')
        print(f'Working directory: {os.getcwd()}')
    except ImportError:
        print('(Drive mount skipped — restart session first if on Colab.)')
elif RUN_ENV == 'local':
    if LOCAL_PROJECT_DIR is not None:
        os.chdir(LOCAL_PROJECT_DIR)
    print(f'Working directory: {os.getcwd()}')
    print('Local run — package install skipped.')
    print('One-time install (Windows/CPU): pip install "jax[cpu]" dm-haiku optax flax')
    print('One-time install (Linux/WSL2 CUDA 12): pip install "jax[cuda12]" dm-haiku optax flax')
    print('GPU check: python -c "import jax; print(jax.devices())"')
    print('NOTE: Use the Python 3.13 (JAX) kernel — JAX does not support Python 3.14.')
else:
    raise ValueError(f'Unknown RUN_ENV: {RUN_ENV!r}  — set to "colab" or "local"')


In [ ]:
import jax
import jax.numpy as jnp
import haiku as hk
import optax
import matplotlib.pyplot as plt
import pickle, json, os, time

from budget_auction.core import (
    BudgetSimulator, BoundedContPolicy,
    build_bounded_policy_net, plot_loss_dict,
)
from budget_auction.mechanisms.budget_mechanism import BudgetMechanism
from budget_auction.mechanisms.input_adapters import VARIANT_NETWORK
from budget_auction.envs.private_budget_multigood_env import create_private_budget_multigood_auction
from budget_auction.train.run_acceptance import nested_GD_budget
from budget_auction.train.evaluate import eval_budget_mechanism_multikey, print_eval_report
from budget_auction.train.binary_search import (
    DEFAULT_BS_CFG, train_with_checkpoints,
    evaluate_checkpoint_list, run_binary_search,
)

print('JAX devices:', jax.devices())


## Configuration

Set `VARIANT` to the mechanism input variant you want to search over. Run one variant per session (each takes ~6-10 hours for a full binary search).

The base hyperparameters below are seeded from the HP search results (Stage 1 winner: `lr_theta=3e-4`, `lr_w=1e-3`; Stage 2: moderate weights). Override `hp_json_path` to load your own HP search results.

In [ ]:
VARIANT  = 'A'   # 'A' | 'B' | 'C'  ← change this per run
SEED     = 42
TIME_BUDGET_HOURS = 11.5   # total wall-clock budget

# ---- Set FORCE_RESTART=True to ignore existing results and start fresh ----
# Use a different bs_save_dir below when changing configs, so old results
# are not overwritten.  FORCE_RESTART=True with the same dir discards them.
FORCE_RESTART = False

# ---- Load HP search best config (optional) ----
hp_json_path = os.path.join('outputs/private_budget_multigood/hparam_search', 'best_hparams.json')
if os.path.exists(hp_json_path):
    with open(hp_json_path) as _f:
        _hp = json.load(_f)
    _opt = _hp.get('optimal_config', {})
    print(f'Loaded HP search config from: {hp_json_path}')
    print('  Optimal config:', _opt)
else:
    _opt = {}
    print(f'HP search file not found at {hp_json_path} — using defaults.')

# ---- Binary search config ----
# Seeded from HP search; override any key as needed.
bs_cfg = {
    **DEFAULT_BS_CFG,
    # Learning rates (from HP search Stage 1)
    'lr_theta':        _opt.get('lr_theta',        3e-4),
    'lr_w':            _opt.get('lr_w',            1e-3),
    # Loss weights at pg_base (from HP search Stage 2)
    'alpha_base':      _opt.get('alpha_weight',    1e5),
    'beta_weight':     _opt.get('beta_weight',     500.0),
    # Inner oracle steps (from HP search Stage 3)
    'num_inner_steps': _opt.get('num_inner_steps', 300),
    # Full training epochs per binary-search trial
    'num_outer_epochs': 2000,
    # Checkpoint recording
    'checkpoint_thresh':    0.15,   # record when train IC+IR both ≤ this
    'checkpoint_interval':  50,     # check every N outer epochs
    # Binary search range
    'pg_lo':   1.5,
    'pg_hi':   3.5,
    'pg_base': 1.5,   # reference point for alpha scaling
    'pg_tol':  0.1,
    # Feasibility thresholds (eval)
    'ic_thresh':  0.05,
    # Revenue gap threshold: checkpoint must achieve revenue >= profit_goal - gap_thresh.
    # Prevents a low-revenue IC solution from falsely pushing pg_lo upward.
    # Default matches pg_tol so 'feasible' means within binary-search resolution.
    'gap_thresh': 0.1,
    # IC re-optimisation at eval time.
    # The oracle is re-solved from scratch against each checkpoint mechanism
    # to get an honest (upper-bound) regret rather than reusing the
    # underconverged training oracle.
    'ic_reopt_steps':    1000,   # inner steps per restart
    'ic_reopt_restarts': 3,      # independent random restarts (take max)
    'ic_reopt_lr':       3e-3,   # Adam lr for reopt (independent of training lr_w)
    'ic_reopt_samples':  128,    # MC samples during reopt ascent
    # Evaluation (revenue, IR, budget-viol, final IC)
    'eval_num_keys':    32,
    'eval_num_samples': 256,
}

print(f'Binary search config for Variant {VARIANT}  (FORCE_RESTART={FORCE_RESTART}):')
for _k, _v in bs_cfg.items():
    print(f'  {_k:25s}: {_v}')


## Build Auction Environment

In [ ]:
auction = create_private_budget_multigood_auction(
    n_buyers=2, n_goods=2, horizon=5, discount=0.9,
    initial_budget=3.0, seed=SEED,
)
print(f'Auction: {auction.num_buyers}B x {auction.num_goods}G x T={auction.num_horizons}')
print(f'Initial budgets: {auction.initial_budgets}')
print(f'Type space: [{auction.types_low}, {auction.types_high}]')


## Build Networks

In [ ]:
mechanism_network = hk.without_apply_rng(hk.transform(VARIANT_NETWORK[VARIANT]))
policy_network = build_bounded_policy_net(
    d_priv=auction.d_priv, d_pub=auction.d_pub,
    G=auction.num_goods, hidden=128,
    residual_init_std=1.0,
)
print(f'Variant {VARIANT} networks built.')
print(f'  d_priv={auction.d_priv}  d_pub={auction.d_pub}  G={auction.num_goods}')


In [ ]:
# To start a brand-new search with a different config, change the RUN_TAG below.
# Each unique tag gets its own subdirectory, so old results are never overwritten.
# Leave RUN_TAG='' to use the default directory.
RUN_TAG = ''   # e.g. 'run2', 'higherLR', 'moreSteps'
_bs_base = f'outputs/private_budget_multigood/binary_search/variant_{VARIANT}'
bs_save_dir = f'{_bs_base}/{RUN_TAG}' if RUN_TAG else _bs_base
os.makedirs(bs_save_dir, exist_ok=True)
print(f'Save directory: {bs_save_dir}')
print(f'Results will be written to: {bs_save_dir}/bs_results.json (fault-tolerant)')
print(f'Best mechanisms: {bs_save_dir}/best_mech_pg<X>.pkl')
print(f'(Set FORCE_RESTART=True above to discard existing results in this directory)')


## Run Binary Search

This cell runs the full binary search. It will:
1. Bisect the `[pg_lo, pg_hi]` interval until `pg_hi - pg_lo < pg_tol`
2. For each midpoint, train a full mechanism and record checkpoints
3. Evaluate all checkpoints on 32 held-out keys × 256 samples
4. Declare feasibility if any checkpoint achieves IC ≤ 0.05 **AND** revenue ≥ profit_goal − gap_thresh
5. Save results after every trial (safe to interrupt and resume)

Expected number of bisection steps: `ceil(log2((pg_hi-pg_lo)/pg_tol))` ≈ 4-5 trials.
Each trial takes ~30-60 min on GPU. Total wall time: up to 12h.

In [ ]:
bs_summary = run_binary_search(
    auction           = auction,
    mechanism_network = mechanism_network,
    policy_network    = policy_network,
    variant           = VARIANT,
    save_dir          = bs_save_dir,
    cfg               = bs_cfg,
    base_seed         = SEED,
    verbose           = True,
    time_budget_hours = TIME_BUDGET_HOURS,
    force_restart     = FORCE_RESTART,
)

print('\n' + '='*60)
print(f'Binary search complete — Exp 2 (Multi-Good Privately Correlated) Variant {VARIANT}')
print(f'  Highest confirmed feasible profit_goal: {bs_summary["pg_lo"]:.3f}')
print(f'  pg_hi (infeasible upper bound): {bs_summary["pg_hi"]:.3f}')
print(f'  Converged: {bs_summary["converged"]}')
if bs_summary['best_mechanism']:
    _bm = bs_summary['best_mechanism']
    print(f'  Best mechanism: rev={_bm["revenue_eval"]:.4f}  '
          f'IC={_bm["regret_eval"]:.4f}  surplus={_bm.get("buyer_surplus_eval", float("nan")):.4f}')
    print(f'  Saved to: {_bm["mech_pkl"]}')


## Analyse Results

Load the saved trial records and visualise the binary search progression.

In [ ]:
_bs_path = os.path.join(bs_save_dir, 'bs_results.json')
with open(_bs_path) as _f:
    _bs_trials = json.load(_f)
_bs_trials_sorted = sorted(_bs_trials, key=lambda t: t['profit_goal'])

print(f'Loaded {len(_bs_trials)} trial records from {_bs_path}')
print(f'\n{"profit_goal":>12} | {"feasible":>8} | {"best_rev":>9} | {"best_IC":>8} | {"surplus":>9} | {"ckpts":>5}')
print('-' * 65)
for _t in _bs_trials_sorted:
    _flag = 'PASS' if _t.get('any_feasible') else 'fail'
    _rev  = _t.get('best_revenue_eval')       or float('nan')
    _ic   = _t.get('best_regret_eval')         or float('nan')
    _srp  = _t.get('best_buyer_surplus_eval')  or float('nan')
    _nc   = _t.get('num_checkpoints', 0)
    print(f'{_t["profit_goal"]:>12.3f} | {_flag:>8} | {_rev:>9.4f} | {_ic:>8.4f} | {_srp:>9.4f} | {_nc:>5}')


In [ ]:
# Plot binary search progression (one point per bisection trial)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

_pgs  = [t['profit_goal'] for t in _bs_trials_sorted]
_revs = [t.get('best_revenue_eval')       or 0.0 for t in _bs_trials_sorted]
_ics  = [t.get('best_regret_eval')         or 1.0 for t in _bs_trials_sorted]
_srps = [t.get('best_buyer_surplus_eval')  or 0.0 for t in _bs_trials_sorted]
_cols = ['tab:green' if t.get('any_feasible') else 'tab:red' for t in _bs_trials_sorted]

axes[0].scatter(_pgs, _revs, c=_cols, s=80, zorder=3)
axes[0].set_xlabel('profit_goal'); axes[0].set_ylabel('Best feasible revenue')
axes[0].set_title('Revenue vs profit_goal')
axes[0].axhline(0, color='k', lw=0.5)

axes[1].scatter(_pgs, _ics, c=_cols, s=80, zorder=3)
axes[1].axhline(0.05, color='red', linestyle='--', label='IC thresh')
axes[1].set_xlabel('profit_goal'); axes[1].set_ylabel('Best IC regret (eval)')
axes[1].set_title('IC Regret vs profit_goal'); axes[1].legend()

axes[2].scatter(_pgs, _srps, c=_cols, s=80, zorder=3)
axes[2].set_xlabel('profit_goal'); axes[2].set_ylabel('Buyer surplus (eval)')
axes[2].set_title('Buyer Surplus vs profit_goal')

from matplotlib.patches import Patch
_legend = [Patch(color='tab:green', label='feasible'), Patch(color='tab:red', label='infeasible')]
axes[0].legend(handles=_legend)
plt.suptitle(f'Exp 2 (Multi-Good Privately Correlated) — Variant {VARIANT} Binary Search', y=1.02)
plt.tight_layout(); plt.show()


## Convergence Plots per Trial

For each binary-search trial, show:
- **Full training curves** (IC regret, IR loss, profit gap) vs epoch — solid lines
- **Checkpoint markers** — triangles at epochs where training IC+IR ≤ threshold
- **Eval values** at checkpoints — circles (filled=feasible, open=infeasible)
- **Dashed red lines** at the 0.05 feasibility threshold

This lets you see whether the mechanism converges, when good checkpoints appear, and whether training metrics are predictive of eval metrics.

In [ ]:
# Load training curves for all trials
_curves_by_pg = {}
for _t in _bs_trials_sorted:
    _cp = _t.get('training_curves')
    if _cp and os.path.exists(_cp):
        with open(_cp) as _f:
            _curves_by_pg[_t['profit_goal']] = json.load(_f)
    else:
        print(f'  [missing] training_curves for pg={_t["profit_goal"]:.3f}')
print(f'Loaded training curves for {len(_curves_by_pg)} trials.')


In [ ]:
# One figure per trial: 3 panels (IC, IR, profit gap)
# IC panel shows three series:
#   - solid line  : full training regret curve
#   - triangle (▲): training value at each checkpoint
#   - circle (●/x) : re-optimised oracle regret (honest eval); green=pass, red=fail
_THRESH = 0.05
for _pg, _cv in sorted(_curves_by_pg.items()):
    _trial = next(t for t in _bs_trials if abs(t['profit_goal'] - _pg) < 1e-6)
    _feasible_trial = _trial.get('any_feasible', False)
    _title_flag = 'FEASIBLE' if _feasible_trial else 'INFEASIBLE'
    _epochs = list(range(len(_cv['reg'])))

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # ---- Panel data definitions ----
    # (label, full_train_curve, ckpt_train_key, ckpt_eval_key, reopt_key, color)
    # Note: IR loss is always 0 by construction (payment feasibility transform),
    # so we show Buyer Surplus (sum of discounted rewards) instead.
    _panels = [
        ('IC Regret',    _cv['reg'], 'reg_train', 'regret_eval',       'regret_reopt', 'tab:blue'),
        ('Buyer Surplus', None,       'ir_train',  'buyer_surplus_eval', None,          'tab:orange'),
        ('Profit Gap',   _cv['gap'], 'gap_train', None,                 None,          'tab:purple'),
    ]

    for _ax, (_label, _train_curve, _ck_train_key, _ck_eval_key, _ck_reopt_key, _col) in zip(axes, _panels):
        # Full training curve (None for buyer surplus — no training-time metric)
        if _train_curve is not None:
            _ax.plot(_epochs, _train_curve, color=_col, lw=1.2, alpha=0.75, label='train')

        # Feasibility threshold lines: IC panel uses ic_thresh, Gap panel uses gap_thresh
        if _ck_eval_key == 'regret_eval':
            _ax.axhline(_THRESH, color='red', linestyle='--', lw=1.0,
                        alpha=0.8, label=f'IC thresh={_THRESH}')
        if _ck_train_key == 'gap_train':
            _ax.axhline(bs_cfg.get('gap_thresh', 0.1), color='red', linestyle='--', lw=1.0,
                        alpha=0.8, label=f'gap thresh={bs_cfg.get("gap_thresh", 0.1)}')

        # Checkpoint training values (triangles)
        _ck_eps   = [c['epoch']       for c in _cv['checkpoints_train']]
        _ck_train = [c[_ck_train_key] for c in _cv['checkpoints_train']]
        if _ck_eps:
            _ax.scatter(_ck_eps, _ck_train, marker='^', s=60,
                        color=_col, zorder=4, label='ckpt (train)')

        # Checkpoint eval values: re-optimised IC (diamond) or multi-key mean (circle)
        if _ck_eval_key is not None:
            for _er in _cv['checkpoints_eval']:
                _col_pt = 'tab:green' if _er['feasible'] else 'tab:red'
                # Re-opt regret (diamond) — this is what determines feasibility
                if _ck_reopt_key and _er.get(_ck_reopt_key) is not None:
                    _ax.scatter(_er['epoch'], _er[_ck_reopt_key], marker='D', s=70,
                                color=_col_pt, zorder=6,
                                label='reopt IC (pass)' if _er['feasible'] else 'reopt IC (fail)')
                # Multi-key eval mean (circle) — shown for comparison
                if _er.get(_ck_eval_key) is not None:
                    _ax.scatter(_er['epoch'], _er[_ck_eval_key], marker='o', s=45,
                                color=_col_pt, alpha=0.5, zorder=5,
                                label='eval mean')
        elif _ck_eval_key == 'buyer_surplus_eval':
            # Buyer surplus: eval circles only (no training curve, no reopt)
            for _er in _cv['checkpoints_eval']:
                _col_pt = 'tab:green' if _er['feasible'] else 'tab:red'
                if _er.get('buyer_surplus_eval') is not None:
                    _mk = 'o' if _er['feasible'] else 'x'
                    _ax.scatter(_er['epoch'], _er['buyer_surplus_eval'], marker=_mk, s=65,
                                color=_col_pt, zorder=5, label='eval surplus')

        _ax.set_xlabel('Epoch'); _ax.set_ylabel(_label)
        _ax.set_title(_label)
        # Deduplicate legend entries
        _hs, _ls = _ax.get_legend_handles_labels()
        _seen = {}; [_seen.setdefault(_l, _h) for _h, _l in zip(_hs, _ls)]
        _ax.legend(list(_seen.values()), list(_seen.keys()), fontsize=8)

    plt.suptitle(f'Exp 2 (Multi-Good Privately Correlated) Variant {VARIANT} | pg={_pg:.3f} — {_title_flag}', y=1.02)
    plt.tight_layout(); plt.show()


## Load and Evaluate Best Mechanism

In [ ]:
_bm_info = bs_summary.get('best_mechanism') if 'bs_summary' in dir() else None

# Alternatively, find the best pkl from trial records
if _bm_info is None:
    _feas = [t for t in _bs_trials if t.get('any_feasible') and t.get('best_mech_pkl')]
    _feas.sort(key=lambda t: t.get('best_revenue_eval') or 0.0, reverse=True)
    _bm_info = _feas[0] if _feas else None

if _bm_info is None:
    print('No feasible mechanism found yet — search may need more iterations.')
else:
    _pkl_path = _bm_info.get('mech_pkl') or _bm_info.get('best_mech_pkl')
    print(f'Loading best mechanism from: {_pkl_path}')
    with open(_pkl_path, 'rb') as _f:
        _best = pickle.load(_f)

    _best_mnet = hk.without_apply_rng(hk.transform(VARIANT_NETWORK[VARIANT]))
    _best_pnet = build_bounded_policy_net(
        d_priv=auction.d_priv, d_pub=auction.d_pub,
        G=auction.num_goods, hidden=128, residual_init_std=1.0,
    )

    _eval_keys = jax.random.split(jax.random.PRNGKey(55555), 64)
    _final_ev = eval_budget_mechanism_multikey(
        auction=auction, mechanism_network=_best_mnet,
        mechanism_params=_best['mechanism_params'],
        profit_goal=_best['profit_goal'],
        num_samples=512, keys=_eval_keys,
        policy_network=_best_pnet, policy_params=_best['policy_params'],
    )
    print_eval_report(_final_ev, prefix=f'Exp 2 (Multi-Good Privately Correlated) Variant {VARIANT} Best: ')
    print(f'  profit_goal: {_best["profit_goal"]:.3f}')
    print(f'  epoch found: {_best["epoch"]}')


## Save Summary

In [ ]:
_summary_path = os.path.join(f'outputs/private_budget_multigood/binary_search/variant_{VARIANT}', f'variant_{VARIANT}_summary.json')
if _bm_info is not None:
    _out = {
        'variant':         VARIANT,
        'exp':             'Exp 2 (Multi-Good Privately Correlated)',
        'pg_lo':           bs_summary.get('pg_lo'),
        'pg_hi':           bs_summary.get('pg_hi'),
        'converged':       bs_summary.get('converged'),
        'total_time_h':    bs_summary.get('total_time_h'),
        'best_mechanism':  {
            'profit_goal':    _best['profit_goal'],
            'epoch':          _best['epoch'],
            'revenue_eval':   float(_final_ev['profit_stats']['mean']),
            'revenue_std':    float(_final_ev['profit_stats']['std']),
            'regret_eval':    float(_final_ev.get('regret_stats', {}).get('mean', float('nan'))),
            'buyer_surplus_eval': float(_final_ev['buyer_surplus_stats']['mean']),
            'budget_viol':    float(_final_ev['budget_viol_stats']['mean']),
            'ir_viol':        float(_final_ev['ir_viol_stats']['mean']),
        },
        'bs_cfg': {k: v for k, v in bs_cfg.items()},
    }
    with open(_summary_path, 'w') as _f:
        json.dump(_out, _f, indent=2)
    print(f'Summary saved to: {_summary_path}')
else:
    print('No feasible mechanism — nothing to save.')
